# Week 2 Day 6 — Dashboard Data Preparation

## ICU Early Warning Prediction System

In Week 2 Day 5, patient-level predictions became explainable using SHAP.

Today, the objective is to prepare clean outputs that can later be used in a dashboard.

A dashboard should display:

- total number of patients analyzed
- number of low, moderate, high, and critical risk patients
- average predicted sepsis risk
- highest-risk patients
- patient-level prediction table
- clinical alert levels

This notebook creates dashboard-ready CSV files from the saved model.

In [1]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
DASHBOARD_DIR = PROJECT_ROOT / "dashboard_data"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dashboard data path:", DASHBOARD_DIR)
print("Dashboard directory exists:", DASHBOARD_DIR.exists())

Project root: c:\Users\User\OneDrive\Desktop\icu-early-warning-system
Dashboard data path: c:\Users\User\OneDrive\Desktop\icu-early-warning-system\dashboard_data
Dashboard directory exists: True


In [3]:
model = joblib.load(MODELS_DIR / "gradient_boosting_smote_model.joblib")
feature_names = joblib.load(MODELS_DIR / "model_feature_names.joblib")
metadata = joblib.load(MODELS_DIR / "model_metadata.joblib")

print("Model loaded successfully.")
print("Model name:", metadata["model_name"])
print("Number of features:", len(feature_names))

Model loaded successfully.
Model name: Gradient Boosting + SMOTE
Number of features: 16


In [4]:
df = pd.read_csv(DATA_PATH / "day2_patient_level_features.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (100, 18)


,Patient_ID,HR_mean,HR_max,HR_min,O2Sat_mean,O2Sat_min,Temp_mean,Temp_max,SBP_mean,SBP_min,MAP_mean,MAP_min,Resp_mean,Resp_max,Age_first,Gender_first,ICULOS_max,SepsisLabel_max
0,p000001,101.907407,117.0,76.0,91.453704,85.0,36.735185,37.44,127.870370,78.0,88.321111,44.00,24.555556,32.0,83.14,0,54,0
1,p000002,62.173913,94.0,54.0,97.043478,94.0,36.206087,37.00,129.043478,114.0,67.239130,50.50,14.630435,27.0,75.91,0,23,0
2,p000003,79.968750,93.0,68.0,95.375000,91.0,37.465000,38.61,139.760417,121.0,81.149167,62.67,25.302083,40.0,45.82,0,48,0
3,p000004,102.172414,113.0,89.0,98.189655,95.5,36.463103,37.00,113.017241,90.0,67.063103,34.00,18.758621,26.0,65.71,0,29,0
4,p000005,76.604167,88.0,61.0,97.677083,96.0,37.072292,37.33,135.072917,114.0,90.364583,73.00,15.447917,21.0,28.09,1,49,0


In [6]:
target_col = "SepsisLabel_max"

X = df[feature_names]
y = df[target_col]
patient_ids = df["Patient_ID"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (100, 16)
Target shape: (100,)


In [7]:
def categorize_risk(probability):
    risk_score = probability * 100

    if risk_score < 25:
        return "Low Risk"
    elif risk_score < 50:
        return "Moderate Risk"
    elif risk_score < 75:
        return "High Risk"
    else:
        return "Critical Risk"


def generate_clinical_alert(risk_category):
    if risk_category == "Low Risk":
        return "Routine Monitoring"
    elif risk_category == "Moderate Risk":
        return "Increased Monitoring"
    elif risk_category == "High Risk":
        return "Clinical Review Recommended"
    else:
        return "Urgent Clinical Review"

In [8]:
probabilities = model.predict_proba(X)[:, 1]
predictions = model.predict(X)

dashboard_predictions = pd.DataFrame({
    "Patient_ID": patient_ids,
    "True_Sepsis_Label": y,
    "Predicted_Sepsis_Label": predictions,
    "Sepsis_Probability": probabilities
})

dashboard_predictions["Risk_Score_Percent"] = dashboard_predictions["Sepsis_Probability"] * 100
dashboard_predictions["Risk_Category"] = dashboard_predictions["Sepsis_Probability"].apply(categorize_risk)
dashboard_predictions["Clinical_Alert_Level"] = dashboard_predictions["Risk_Category"].apply(generate_clinical_alert)

dashboard_predictions = dashboard_predictions.sort_values(
    by="Risk_Score_Percent",
    ascending=False
)

dashboard_predictions.head(10)

,Patient_ID,True_Sepsis_Label,Predicted_Sepsis_Label,Sepsis_Probability,Risk_Score_Percent,Risk_Category,Clinical_Alert_Level
52,p000053,1,1,0.999427,99.942674,Critical Risk,Urgent Clinical Review
41,p000042,1,1,0.999322,99.932154,Critical Risk,Urgent Clinical Review
27,p000028,1,1,0.999261,99.926142,Critical Risk,Urgent Clinical Review
8,p000009,1,1,0.999256,99.925559,Critical Risk,Urgent Clinical Review
17,p000018,1,1,0.998852,99.885235,Critical Risk,Urgent Clinical Review
33,p000034,1,1,0.998820,99.881973,Critical Risk,Urgent Clinical Review
55,p000056,1,1,0.998790,99.879013,Critical Risk,Urgent Clinical Review
14,p000015,1,1,0.998649,99.864915,Critical Risk,Urgent Clinical Review
57,p000058,1,1,0.998407,99.840659,Critical Risk,Urgent Clinical Review
10,p000011,1,1,0.997558,99.755789,Critical Risk,Urgent Clinical Review


In [9]:
dashboard_predictions.to_csv(
    DASHBOARD_DIR / "dashboard_patient_predictions.csv",
    index=False
)

print("Dashboard patient prediction table saved successfully.")

Dashboard patient prediction table saved successfully.


In [10]:
risk_distribution = dashboard_predictions["Risk_Category"].value_counts().reset_index()
risk_distribution.columns = ["Risk_Category", "Number_of_Patients"]

risk_order = ["Low Risk", "Moderate Risk", "High Risk", "Critical Risk"]

risk_distribution["Risk_Category"] = pd.Categorical(
    risk_distribution["Risk_Category"],
    categories=risk_order,
    ordered=True
)

risk_distribution = risk_distribution.sort_values("Risk_Category")

risk_distribution

,Risk_Category,Number_of_Patients
0,Low Risk,81
2,Moderate Risk,2
1,Critical Risk,17


In [11]:
risk_distribution.to_csv(
    DASHBOARD_DIR / "dashboard_risk_distribution.csv",
    index=False
)

print("Dashboard risk distribution saved successfully.")

Dashboard risk distribution saved successfully.


In [12]:
total_patients = len(dashboard_predictions)
average_risk = dashboard_predictions["Risk_Score_Percent"].mean()
max_risk = dashboard_predictions["Risk_Score_Percent"].max()
min_risk = dashboard_predictions["Risk_Score_Percent"].min()

predicted_sepsis_count = dashboard_predictions["Predicted_Sepsis_Label"].sum()
true_sepsis_count = dashboard_predictions["True_Sepsis_Label"].sum()

high_or_critical_count = dashboard_predictions[
    dashboard_predictions["Risk_Category"].isin(["High Risk", "Critical Risk"])
].shape[0]

dashboard_summary = pd.DataFrame({
    "Metric": [
        "Total Patients",
        "True Sepsis Cases",
        "Predicted Sepsis Cases",
        "High or Critical Risk Patients",
        "Average Risk Score (%)",
        "Maximum Risk Score (%)",
        "Minimum Risk Score (%)"
    ],
    "Value": [
        total_patients,
        true_sepsis_count,
        predicted_sepsis_count,
        high_or_critical_count,
        round(average_risk, 2),
        round(max_risk, 2),
        round(min_risk, 2)
    ]
})

dashboard_summary

,Metric,Value
0,Total Patients,100.00
1,True Sepsis Cases,14.00
2,Predicted Sepsis Cases,17.00
3,High or Critical Risk Patients,17.00
4,Average Risk Score (%),17.69
5,Maximum Risk Score (%),99.94
6,Minimum Risk Score (%),0.01


In [13]:
dashboard_summary.to_csv(
    DASHBOARD_DIR / "dashboard_summary_metrics.csv",
    index=False
)

print("Dashboard summary metrics saved successfully.")

Dashboard summary metrics saved successfully.


In [14]:
top_10_high_risk_patients = dashboard_predictions.head(10)

top_10_high_risk_patients

,Patient_ID,True_Sepsis_Label,Predicted_Sepsis_Label,Sepsis_Probability,Risk_Score_Percent,Risk_Category,Clinical_Alert_Level
52,p000053,1,1,0.999427,99.942674,Critical Risk,Urgent Clinical Review
41,p000042,1,1,0.999322,99.932154,Critical Risk,Urgent Clinical Review
27,p000028,1,1,0.999261,99.926142,Critical Risk,Urgent Clinical Review
8,p000009,1,1,0.999256,99.925559,Critical Risk,Urgent Clinical Review
17,p000018,1,1,0.998852,99.885235,Critical Risk,Urgent Clinical Review
33,p000034,1,1,0.998820,99.881973,Critical Risk,Urgent Clinical Review
55,p000056,1,1,0.998790,99.879013,Critical Risk,Urgent Clinical Review
14,p000015,1,1,0.998649,99.864915,Critical Risk,Urgent Clinical Review
57,p000058,1,1,0.998407,99.840659,Critical Risk,Urgent Clinical Review
10,p000011,1,1,0.997558,99.755789,Critical Risk,Urgent Clinical Review


In [15]:
top_10_high_risk_patients.to_csv(
    DASHBOARD_DIR / "dashboard_top_10_high_risk_patients.csv",
    index=False
)

print("Top 10 high-risk patients saved successfully.")

Top 10 high-risk patients saved successfully.


In [16]:
dashboard_clinical_view = dashboard_predictions[
    [
        "Patient_ID",
        "Risk_Score_Percent",
        "Risk_Category",
        "Clinical_Alert_Level",
        "Predicted_Sepsis_Label"
    ]
]

dashboard_clinical_view.head(10)

,Patient_ID,Risk_Score_Percent,Risk_Category,Clinical_Alert_Level,Predicted_Sepsis_Label
52,p000053,99.942674,Critical Risk,Urgent Clinical Review,1
41,p000042,99.932154,Critical Risk,Urgent Clinical Review,1
27,p000028,99.926142,Critical Risk,Urgent Clinical Review,1
8,p000009,99.925559,Critical Risk,Urgent Clinical Review,1
17,p000018,99.885235,Critical Risk,Urgent Clinical Review,1
33,p000034,99.881973,Critical Risk,Urgent Clinical Review,1
55,p000056,99.879013,Critical Risk,Urgent Clinical Review,1
14,p000015,99.864915,Critical Risk,Urgent Clinical Review,1
57,p000058,99.840659,Critical Risk,Urgent Clinical Review,1
10,p000011,99.755789,Critical Risk,Urgent Clinical Review,1


In [17]:
dashboard_clinical_view.to_csv(
    DASHBOARD_DIR / "dashboard_clinical_view.csv",
    index=False
)

print("Dashboard clinical view saved successfully.")

Dashboard clinical view saved successfully.


## Dashboard Interpretation

This notebook prepared the main data outputs needed for a future ICU Early Warning dashboard.

The dashboard-ready files include:

- patient-level prediction table
- risk category distribution
- summary metrics
- top 10 highest-risk patients
- simplified clinical view

These files can later be used in a Streamlit dashboard, Flask app, FastAPI backend, or Power BI report.

This step moves the project closer to deployment because the model outputs are now organized in a way that can be directly consumed by an interface.

In [19]:
print("Week 2 Day 6 completed successfully.")
print("Generated dashboard files:")
print("- dashboard_patient_predictions.csv")
print("- dashboard_risk_distribution.csv")
print("- dashboard_summary_metrics.csv")
print("- dashboard_top_10_high_risk_patients.csv")
print("- dashboard_clinical_view.csv")

Week 2 Day 6 completed successfully.
Generated dashboard files:
- dashboard_patient_predictions.csv
- dashboard_risk_distribution.csv
- dashboard_summary_metrics.csv
- dashboard_top_10_high_risk_patients.csv
- dashboard_clinical_view.csv
